# Proof of concept for a RAG System on arxiv abstracts

# Architecture:
- Embeddings: sentence-transformer
- Vectors: FAISS
- PDF parsing with PyMuPDF
- Models from Huggingface

## Imports and Configs

In [2]:
# !pip install kagglehub
# !pip install pymupdf
# !pip install faiss-cpu

In [3]:
import pandas as pd
import pymupdf as fitz
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import faiss

import os, json, re, textwrap, time, requests
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
path = Path("../Data/datasets/arxiv_data.csv")

CFG = {
    "data_path" : path,
    
    # Change these according to final chosen dataset
    "col_title" : "titles",
    "col_summaries" : "summaries",
    "col_terms" : "terms",
    
    "embed_model" : "sentence-transformers/all-MiniLM-L6-v2", # Larger alternative: "sentence-transformers/all-mpnet-base-v2"
    "embed_batch" : 256,
    
    "llm_model" : "Qwen/Qwen2.5-3B-Instruct", # Larger alternative: "Qwen/Qwen2.5-7B-Instruct", "meta-llama/Llama-3.1-8B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
    "llm_max_tokens" : 512,
    "llm_temperature" : 0.7,
    "llm_load_in_4bit" : False,
    
    "top_k" : 5,
    
    # -- Caches -- (to avoid recomputation during development)
    "index_path" : "arxiv.faiss", # FAISS index file
    "meta_path" : "arxiv_meta.json", # Metadata for each paper (title, summary, terms, embedding vector)
}

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # Non M-series Mac
DEVICE = "mps" if torch.backends.mps.is_available() else DEVICE # M-series Mac
print(f"Using device: {DEVICE}")

Using device: mps


## Preprocessing of Dataset

In [5]:
def load_dataset(cfg):
    path = cfg["data_path"]
    df = pd.read_csv(path)
    
    df.columns = [c.strip().lower() for c in df.columns]
    required = [cfg["col_title"], cfg["col_summaries"]] # Change this according to final chosen dataset
    
    df = df.dropna(subset=required).drop_duplicates(subset=cfg["col_title"]).reset_index(drop=True)
    df["all_text"] = (
        df[cfg["col_title"]].str.strip() + " " + df[cfg["col_summaries"]].str.strip()
    )
    
    print(f"Loaded {len(df)} papers.")
    return df


df = load_dataset(CFG)
df.head(2)

Loaded 38972 papers.


,titles,summaries,terms,all_text
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']",Survey on Semantic Stereo Matching / Semantic ...
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']",FUTURE-AI: Guiding Principles and Consensus Re...


## Embeddings

In [6]:
def load_embed_model(cfg):
    model = SentenceTransformer(cfg["embed_model"], device=DEVICE)
    print(f"  Embedding dim: {model.get_sentence_embedding_dimension()}")
    return model

embed_model = load_embed_model(CFG)

  Embedding dim: 384


## FAISS Index

Used for similarity search and clustering of vectors

In [7]:
def embed_texts(texts, model, batch_size):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i : i + batch_size]
        embs = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
        embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10) # Normalize 
        all_embeddings.append(embs.astype(np.float32))
        
    return np.vstack(all_embeddings)

In [8]:
def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner Product for cosine similarity
    index.add(embeddings)
    print(f"FAISS index built with {index.ntotal} vectors.")
    
    return index


def save_index(index, meta, cfg):
    faiss.write_index(index, cfg["index_path"])
    with open(cfg["meta_path"], "w") as f:
        json.dump(meta, f)
    print(f"Index and metadata saved to {cfg['index_path']} and {cfg['meta_path']}.")
    

def load_index(cfg):
    index = faiss.read_index(cfg["index_path"])
    with open(cfg["meta_path"], "r") as f:
        meta = json.load(f)
    print(f"Index and metadata loaded from {cfg['index_path']} and {cfg['meta_path']}.")
    return index, meta

In [9]:
def get_or_build_index(df, embed_model, cfg):
    if Path(cfg["index_path"]).exists() and Path(cfg["meta_path"]).exists():
        print("Found existing index and metadata. Loading...")
        return load_index(cfg)
    
    print("No existing index found. Building new index...")
    texts = df["all_text"].tolist()
    embeddings = embed_texts(texts, embed_model, cfg["embed_batch"])

    index = build_faiss_index(embeddings)
    
    meta = [
        {
            "idx" : int(i),
            "title" : str(row[cfg["col_title"]]),
            "summary" : str(row[cfg["col_summaries"]]),
            "terms" : str(row[cfg["col_terms"]]) if cfg["col_terms"] in row else "",
        }
        for i, row in df.iterrows()
    ]
    
    save_index(index, meta, cfg)
    return index, meta

index, meta = get_or_build_index(df, embed_model, CFG)

Found existing index and metadata. Loading...
Index and metadata loaded from arxiv.faiss and arxiv_meta.json.


## Retrieval

In [10]:
def retrieve(query, embed_model, index, meta, top_k):
    """
    Embed a query string and return the top-k most similar papers.
    Each result dict contains: title, abstract, terms, score.
    """
    
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-10) # Normalize
    
    scores, ids = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        entry = meta[idx].copy()
        entry["score"] = float(score)
        results.append(entry)
    return results

# Quick sanity check
sample = retrieve("transformer attention mechanism NLP", embed_model, index, meta, top_k=3)
for r in sample:
    print(f"[{r['score']:.3f}] {r['title']}")

[0.702] Adaptive Attention Span in Transformers
[0.697] ETC: Encoding Long and Structured Inputs in Transformers
[0.688] Attention mechanisms and deep learning for machine vision: A survey of the state of the art


## LLM for answer generation

In [11]:
def load_llm(cfg):
    model_id = cfg["llm_model"]
    print(f"Loading LLM: {model_id} (This might take a while...)")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    load_kwargs = {"torch_dtype": torch.float16 if DEVICE == "mps" else torch.float32} # M-series Mac
    # load_kwargs = {"torch_dtype": torch.float16 if DEVICE == "cuda" else torch.float32} # Non M-series Mac
    
    if DEVICE == "mps": # or cuda
        load_kwargs["device_map"] = "auto"
    if cfg.get("llm_load_in_4bit") and DEVICE == "mps": # or cuda
        from transformers import BitsAndBytesConfig
        load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)
        load_kwargs.pop("torch_dtype", None)

    model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)
    if DEVICE == "cpu":
        model = model.to(DEVICE)

    model.eval()
    print("LLM loaded.")
    return tokenizer, model

llm_tokenizer, llm_model = load_llm(CFG)

Loading LLM: Qwen/Qwen2.5-3B-Instruct (This might take a while...)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.78s/it]


LLM loaded.


### LLM sanity checks (cause i'm losing mine)

In [12]:
def demo_llm_generation(tokenizer, model, device):
    DEMOS = [
        {
            "label" : " A factual recall",
            "prompt" : "What is a transformer in the context of machine learning? Answer in 2-3 sentences.",
        },
        {
            "label" : "Instruction following",
            "prompt" : "List exactly three advantages of self-attention over recurrent neural networks. Use a numbered list."
        },
        {
            "label" : "Scientific Reasoning",
            "prompt" : "A language model is trained on a large text corpus and then fine-tuned on a small labelled dataset. What is this training strategy called and why does it work? Be concise.",
        },
    ]
    
    for demo in DEMOS:
        try:
            messages = [{"role": "user", "content": demo["prompt"]}]
            input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
        except Exception:
            input_ids = tokenizer(demo["prompt"], return_tensors="pt").input_ids.to(device)
        
        with torch.no_grad():
            out = model.generate(
                input_ids,
                max_new_tokens = 150,
                temperature = 0.0,
                do_sample = False,
                pad_token_id = tokenizer.eos_token_id,
            )
            
        new_tokens = out[0][input_ids.shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        n_tok = len(new_tokens)
        
        print(f"Demo: {demo['label']}")
        print(f"Prompt: {demo['prompt']}")
        print(f"Response: {response}")
        print(f"Number of tokens generated: {n_tok}")
        print("\n\n")

demo_llm_generation(llm_tokenizer, llm_model, DEVICE)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Demo:  A factual recall
Prompt: What is a transformer in the context of machine learning? Answer in 2-3 sentences.
Response: A transformer is a type of deep learning model that is particularly effective for natural language processing tasks due to its ability to handle sequences of data without relying on the sequential order, thanks to its self-attention mechanism. This allows transformers to understand the importance of different parts of a sequence relative to each other, making them powerful for applications like text generation, translation, and sentiment analysis.
Number of tokens generated: 76



Demo: Instruction following
Prompt: List exactly three advantages of self-attention over recurrent neural networks. Use a numbered list.
Response: 1. **Efficiency in Parallel Processing**: Self-attention mechanisms can process all elements of the input sequence simultaneously, which allows for efficient parallel computation. This is particularly advantageous when dealing with long seque

## Prompt builder & Generation

In [13]:
def build_prompt(query, retrieved):
    context_blocks = []
    for i, r in enumerate(retrieved, 1):
        context_blocks.append(
            f"[Paper {i}]\n"
            f"Title   : {r['title']}\n"
            f"Terms   : {r.get('terms','N/A')}\n"
            f"Abstract: {r['summary'][:600]}{'…' if len(r['summary'])>600 else ''}"
        )
    context = "\n\n".join(context_blocks)

    prompt = (
        "You are a helpful scientific assistant. "
        "Use ONLY the papers provided below to answer the user's question. "
        "Cite papers by their number, e.g. [Paper 1]. "
        "If the answer cannot be found in the provided papers, say so.\n\n"
        "=== RETRIEVED PAPERS ===\n"
        f"{context}\n\n"
        "=== USER QUESTION ===\n"
        f"{query}\n\n"
        "=== YOUR ANSWER ===\n"
    )
    return prompt

def generate_answer(prompt, tokenizer, model, cfg):
    try:
        messages = [{"role": "user", "content": prompt}]
        inputs   = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
        ).to(DEVICE)
        input_ids = inputs
    except Exception:
        # Fallback: tokenise raw prompt
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens  = cfg["llm_max_tokens"],
            temperature     = cfg["llm_temperature"],
            do_sample       = cfg["llm_temperature"] > 0,
            pad_token_id    = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    answer     = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer

### Sanity Checks

In [14]:
def demo_prompt_and_generation(tokenizer, model, cfg: dict) -> None:
    """
    Build a realistic RAG prompt from two well-known NLP papers and
    display the full prompt + the full LLM response side-by-side.
    This test was generated by Claude
    """
    # ── Two representative "retrieved" papers (hard-coded, no retrieval needed) ──
    MOCK_RETRIEVED = [
        {
            "title"   : "Attention Is All You Need",
            "terms"   : "cs.CL cs.LG",
            "summary": (
                "The dominant sequence transduction models are based on complex "
                "recurrent or convolutional neural networks that include an encoder "
                "and a decoder. The best performing models also connect the encoder "
                "and decoder through an attention mechanism. We propose a new simple "
                "network architecture, the Transformer, based solely on attention "
                "mechanisms, dispensing with recurrence and convolutions entirely. "
                "Experiments on two machine translation tasks show these models to be "
                "superior in quality while being more parallelisable and requiring "
                "significantly less time to train."
            ),
            "score"   : 0.963,
        },
        {
            "title"   : "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
            "terms"   : "cs.CL",
            "summary": (
                "We introduce a new language representation model called BERT, which "
                "stands for Bidirectional Encoder Representations from Transformers. "
                "Unlike recent language representation models, BERT is designed to "
                "pre-train deep bidirectional representations from unlabelled text by "
                "jointly conditioning on both left and right context in all layers. "
                "As a result, the pre-trained BERT model can be fine-tuned with just "
                "one additional output layer to create state-of-the-art models for a "
                "wide range of tasks."
            ),
            "score"   : 0.891,
        },
    ]

    TEST_QUERY = (
        "What are the main differences between the Transformer architecture "
        "and BERT, and how does pre-training help in NLP tasks?"
    )


    # ── 1. Build and display the prompt ─────────────────────────────────────────
    prompt = build_prompt(TEST_QUERY, MOCK_RETRIEVED)

    print("\n")
    print("  OUTPUT DEMO -- Section 9: Assembled RAG Prompt")
    print("\n")
    print(f"\n  Query : {TEST_QUERY}\n")
    print("  -- Full prompt sent to LLM " + "-" * 45)
    for line in prompt.split("\n"):
        print("  " + line)
    print("  ")
    print(f"  Total prompt length: {len(prompt)} chars / "
        f"~{len(tokenizer.encode(prompt))} tokens\n")

    # ── 2. Generate and display the answer ──────────────────────────────────────
    print("\n")
    print("  OUTPUT DEMO -- Section 9: LLM Generated Answer")
    print("\n")
    print(f"\n  Generating answer (max_new_tokens={cfg['llm_max_tokens']}) ...\n")

    answer  = generate_answer(prompt, tokenizer, model, cfg)

    print("  -- Answer " + "-" * 62)
    for para in answer.split("\n"):
        if para.strip():
            for line in textwrap.wrap(para, width=68):
                print("  " + line)
        else:
            print()
            
    print()
    print(f"  -- Stats: {len(answer)} chars | "
        f"{len(tokenizer.encode(answer))} tokens generated | ")
    print("\n")

    # ── 3. Spot-check: does the answer cite at least one paper? ─────────────────
    cited = any(f"Paper {i}" in answer or f"paper {i}" in answer
                for i in range(1, len(MOCK_RETRIEVED) + 1))
    cite_str = ("PASS  Answer contains at least one [Paper N] citation."
                if cited else
                "WARN  No [Paper N] citation found -- check the prompt format or model.")
    print(f"  {cite_str}\n")

demo_prompt_and_generation(llm_tokenizer, llm_model, CFG)



  OUTPUT DEMO -- Section 9: Assembled RAG Prompt



  Query : What are the main differences between the Transformer architecture and BERT, and how does pre-training help in NLP tasks?

  -- Full prompt sent to LLM ---------------------------------------------
  You are a helpful scientific assistant. Use ONLY the papers provided below to answer the user's question. Cite papers by their number, e.g. [Paper 1]. If the answer cannot be found in the provided papers, say so.
  
  === RETRIEVED PAPERS ===
  [Paper 1]
  Title   : Attention Is All You Need
  Terms   : cs.CL cs.LG
  Abstract: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments

# Full Pipeline

In [15]:
def rag_pipeline(
    query,
    embed_model,
    index,
    meta,
    tokenizer,
    llm,
    cfg,
):
    """
    1. Retrieve relevant papers for the query
    2. Build a RAG prompt using the retrieved papers
    3. Generate an answer using the LLM
    4. Return the answer and the retrieved papers
    """
    
    # Step 1: Retrieve
    retrieved = retrieve(query, embed_model, index, meta, top_k = cfg["top_k"])
    
    # Step 2: Build prompt
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt, tokenizer, llm, cfg)
    
    return {
        "query" : query,
        "answer" : answer,
        "retrieved_papers" : retrieved,
    }
    
def display_result(result, wrap):
    print("\n")
    print("  FINAL RAG OUTPUT".center(80, "-"))
    print("\n")
    
    print(f"  Query : {result['query']}\n")
    
    print("  -- Answer " + "-" * 62)
    for para in result["answer"].split("\n"):
        if para.strip():
            for line in textwrap.wrap(para, width=wrap):
                print("  " + line)
        else:
            print()
            
    print()
    print(f"  -- Retrieved Papers (Top {len(result['retrieved_papers'])}) " + "-" * 20)
    for i, paper in enumerate(result["retrieved_papers"], 1):
        title = paper['title']
        score = paper['score']
        terms = paper.get('terms', 'N/A')
        print(f"  [{score:.3f}] Paper {i}: {title} [{terms}]")
    print("\n")
    
    
# Demos:
TEST_QUERIES = [
    "What are the latest advancements in transformer architectures for NLP?",
    "How does BERT differ from traditional RNN-based language models?",
    "What are some effective techniques for fine-tuning large language models on small datasets?",
]

for query in TEST_QUERIES:
    result = rag_pipeline(
        query,
        embed_model,
        index,
        meta,
        llm_tokenizer,
        llm_model,
        CFG,
    )
    display_result(result, wrap=80)



-------------------------------  FINAL RAG OUTPUT-------------------------------


  Query : What are the latest advancements in transformer architectures for NLP?

  -- Answer --------------------------------------------------------------
  The latest advancements in transformer architectures for Natural Language
  Processing (NLP) can be explored through a combination of the papers provided.
  However, it is important to note that the provided papers do not explicitly list
  the latest advancements in transformer architectures. They do cover various
  aspects of transformers including modifications, compression techniques, and
  surveys.

  For the most current developments, it would be advisable to consult recent
  research papers and conferences focused on NLP, such as ACL (Association for
  Computational Linguistics), EMNLP (Empirical Methods in Natural Language
  Processing), NeurIPS (Conference on Neural Information Processing Systems), and
  others. These platforms often high

# PDF Extraction

In [16]:
def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = [page.get_text("text") for page in doc]
    doc.close()
    
    text = "\n".join(pages)
    text = re.sub(r"\n{3, }", "\n\n", text) # Collapse multiple newlines
    text = re.sub(r"[ \t]+", " ", text) # Replace multiple spaces/tabs with single space
    return text.strip()

def pdf_to_query(pdf_path, max_chars = 2000):
    text = extract_pdf_text(pdf_path)
    return text[:max_chars]

# Chatbot Sophisticated

In [17]:
def display_result(result: dict, wrap: int = 100):
    """Pretty-print a RAG result in the notebook."""
    line = "─" * wrap
    print(f"\n{'═'*wrap}")
    print(f"  QUERY: {result['query']}")
    print(f"{'═'*wrap}\n")
 
    print("  GENERATED ANSWER\n" + line)
    for para in result["answer"].split("\n"):
        print(textwrap.fill(para, width=wrap, initial_indent="  ", subsequent_indent="  "))
 
    print(f"\n  TOP PAPERS ({len(result['papers'])} retrieved)\n" + line)
    for i, p in enumerate(result["papers"], 1):
        print(f"\n  [{i}] {p['title']}")
        print(f"       Score : {p['score']:.4f}")
        print(f"       Terms : {p.get('terms','N/A')}")
        if p.get("url"):
            print(f"       URL   : {p['url']}")
        if p.get("authors"):
            names = ", ".join(a["name"] for a in p["authors"])
            print(f"       Authors: {names}")
        abstract_preview = p["abstract"][:250].replace("\n", " ")
        print(f"       Abstract: {abstract_preview}…")
 
    print(f"\n{'═'*wrap}\n")

In [18]:
def search_by_text(query, fetch_authors = True):
    """
    Accepts either:
    - A keyword string   e.g. "graph neural network drug discovery"
    - A full sentence    e.g. "How do transformers handle long-range dependencies?"
    """
    result = rag_pipeline(
        query=query,
        embed_model=embed_model,
        index=index,
        meta=meta,
        tokenizer=llm_tokenizer,
        llm=llm_model,
        cfg=CFG,
    )
    display_result(result)
    return result

In [19]:
def search_by_pdf(pdf_path, max_chars = 2000):
    """
    Extract text from a PDF, use it as the query.
    Typically extracts title + abstract from the first page.
    """
    assert Path(pdf_path).exists(), f"PDF not found: {pdf_path}"
    query = pdf_to_query(pdf_path, max_chars=max_chars)
    print(f"PDF query preview (first 200 chars):\n  {query[:200]}\n")
    return search_by_text(query)

In [20]:
def chatbot(history=None):
    """
    Simple multi-turn chatbot in the notebook.
    Type 'exit' or 'quit' to stop, 'clear' to reset history.
    Type 'pdf <path>' on any turn to attach a PDF as additional context.
    """
    if history is None:
        history = []

    print("arXiv RAG Chatbot  (type 'exit' to quit, 'clear' to reset history)")
    print("Attach a PDF any time by typing:  pdf <path/to/file.pdf>")
    print("─" * 60)

    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nStopped.")
            break

        if not user_input:
            continue
        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye!")
            break
        if user_input.lower() == "clear":
            history = []
            print("History cleared.")
            continue

        # ── PDF attachment ───────────────────────────────────────────────────
        pdf_context = ""
        if user_input.lower().startswith("pdf "):
            pdf_path = user_input[4:].strip()
            if not Path(pdf_path).exists():
                print(f"  [!] File not found: {pdf_path}")
                continue
            print(f"  [+] Reading PDF: {pdf_path} ...")
            pdf_context = pdf_to_query(pdf_path, max_chars=3000)
            print(f"  [+] Extracted {len(pdf_context)} chars. "
                    "What would you like to know about it?")
            # Ask the follow-up question before retrieving
            try:
                user_input = input("\nYou: ").strip()
            except (KeyboardInterrupt, EOFError):
                print("\nStopped.")
                break
            if not user_input:
                continue

        # ── Combine PDF text with the typed query for retrieval ──────────────
        retrieval_query = f"{pdf_context} {user_input}".strip() if pdf_context else user_input

        # ── Retrieve ─────────────────────────────────────────────────────────
        retrieved = retrieve(retrieval_query, embed_model, index, meta, top_k=CFG["top_k"])

        # ── Build system context ─────────────────────────────────────────────
        context_blocks = []
        for i, r in enumerate(retrieved, 1):
            context_blocks.append(
                f"[Paper {i}] {r['title']}\n"
                f"Terms: {r.get('terms', 'N/A')}\n"
                f"Abstract: {r.get('summary', r.get('abstract', ''))[:500]}"
            )

        # Inject PDF text as an extra context block if present
        if pdf_context:
            context_blocks.append(
                f"[Attached PDF]\n{pdf_context[:2000]}"
            )

        system_msg = (
            "You are a helpful scientific assistant. "
            "Answer the user's question using the retrieved papers below"
            + (" and the attached PDF." if pdf_context else ".") +
            " Cite papers by number, e.g. [Paper 1]."
            " If a PDF was attached, refer to it as [Attached PDF].\n\n"
            "=== RETRIEVED PAPERS ===\n"
            + "\n\n".join(context_blocks)
        )

        # ── Build multi-turn messages ─────────────────────────────────────────
        messages = [{"role": "system", "content": system_msg}]
        messages += history
        messages.append({"role": "user", "content": user_input})

        # ── Tokenise ──────────────────────────────────────────────────────────
        try:
            input_ids = llm_tokenizer.apply_chat_template(
                messages, return_tensors="pt", add_generation_prompt=True
            ).to(DEVICE)
        except Exception:
            flat = system_msg + "\n"
            for msg in history:
                flat += f"\n{msg['role'].capitalize()}: {msg['content']}"
            flat += f"\nUser: {user_input}\nAssistant:"
            input_ids = llm_tokenizer(flat, return_tensors="pt").input_ids.to(DEVICE)

        # ── Generate ──────────────────────────────────────────────────────────
        with torch.no_grad():
            out = llm_model.generate(
                input_ids,
                max_new_tokens=CFG["llm_max_tokens"],
                temperature=CFG["llm_temperature"],
                do_sample=CFG["llm_temperature"] > 0,
                pad_token_id=llm_tokenizer.eos_token_id,
            )
        new_tokens = out[0][input_ids.shape[-1]:]
        answer = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        # ── Update history ────────────────────────────────────────────────────
        # Store PDF context inside the user turn so future turns are aware of it
        history_user_content = f"[PDF attached]\n{pdf_context}\n\n{user_input}" \
                            if pdf_context else user_input
        history.append({"role": "user",      "content": history_user_content})
        history.append({"role": "assistant", "content": answer})
        history = history[-20:]

        # ── Print output ──────────────────────────────────────────────────────
        print(f"\nAssistant: {answer}")
        print("\n  Related papers:")
        for i, r in enumerate(retrieved, 1):
            print(f"  [{i}] {r['title']} (score={r['score']:.3f})")

    return history

In [21]:
# Start Chatbot

history = chatbot()

arXiv RAG Chatbot  (type 'exit' to quit, 'clear' to reset history)
Attach a PDF any time by typing:  pdf <path/to/file.pdf>
────────────────────────────────────────────────────────────

Assistant: A directed graph, often referred to as a digraph, is a graph in which edges have directions associated with them. Unlike an undirected graph where edges are bidirectional, each edge in a directed graph points from one vertex to another. This directional property makes directed graphs useful for modeling relationships where directionality is important, such as in social networks, citation networks, and computer networks.

In more formal terms, a directed graph consists of a set of vertices or nodes and a set of arcs or directed edges, where each arc connects a pair of vertices and has a specific direction from one vertex to another.

If you need further details or examples, please let me know!

  Related papers:
  [1] Directed hypergraph neural network (score=0.592)
  [2] edGNN: a Simple and P